# 🦜🔗 LangChain Deep Technical Demo – Innomatics Internship Feb 2026

**Author:** Harsha Devupalli  
**Assignment:** GenAI – LangChain Deep Technical Blog  
**Program:** Innomatics Research Labs – Data Science Internship, February 2026

---

This notebook walks through all the core components of LangChain — from a basic LLM call all the way to RAG and Agents.  
I'm using **Groq's free API** with **Llama3** so there's zero cost involved.

### What we'll cover:
1. Installation & Setup
2. Basic LLM Call + Streaming
3. PromptTemplates
4. Chains (LCEL)
5. Memory (Multi-turn Chat)
6. Agents + Custom Tools
7. Document Loader + Vector Store + RAG
8. Architecture Diagram
9. Summary Table

## 📦 Section 1: Installation

First things first — installing all the libraries we need.  
I kept this minimal on purpose: fewer libraries = fewer version conflict headaches.
- `langchain-groq` → connects LangChain to Groq's blazing-fast API
- `langchain-core` → the base Runnable interface, prompts, parsers
- `langchain-community` → document loaders, Chroma vector store wrapper
- `chromadb` → the actual vector database
- `sentence-transformers` → free local embeddings (no OpenAI needed!)
- `langchainhub` → lets us pull pre-built prompts from LangChain Hub

In [27]:
# Install all required packages
# -q means quiet mode (less output clutter in Colab)
# I ran this once and it took about 2 minutes — totally normal
!pip install -q langchain-groq langchain-core langchain-community chromadb sentence-transformers langchainhub

## 🔑 Section 2: API Key Setup

We use `getpass` so the API key is typed securely and never shows up in the notebook output.  
**Get your free Groq key at:** https://console.groq.com  

⚠️ Never hardcode your API key in a notebook — especially if you're pushing to GitHub!

In [2]:
import os
import getpass

# getpass hides the input so the key doesn't appear in notebook output
# I learned this the hard way after almost pushing my key to GitHub 😅
print("Enter your Groq API Key (get it free at console.groq.com):")
GROQ_API_KEY = getpass.getpass()

# Set it as an environment variable — LangChain reads it automatically
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("✅ API Key set successfully! (hidden for security)")

Enter your Groq API Key (get it free at console.groq.com):
··········
✅ API Key set successfully! (hidden for security)


## 🤖 Section 3: Core Component 1 – LLM / ChatGroq

The foundation of everything. `ChatGroq` wraps Groq's API into LangChain's standard interface.

**Why this matters:** If you ever want to switch from Groq to OpenAI, you just swap `ChatGroq` for `ChatOpenAI` — the rest of your code stays the same. That's the whole point of LangChain's abstraction layer.

We're using `llama-3.3-70b-versatile` — it's Groq's latest Llama3.3 model and it's **free** with generous rate limits.

In [3]:
from langchain_groq import ChatGroq

# Initialize the LLM
# temperature=0 means deterministic (same question = same answer)
# temperature=0.7 would be more creative/varied
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # Groq's latest Llama3.3 model
    temperature=0,
    max_tokens=1024
)

print("✅ ChatGroq initialized with Llama3.3-70B")
print(f"   Model: {llm.model_name}")

✅ ChatGroq initialized with Llama3.3-70B
   Model: llama-3.3-70b-versatile


In [4]:
# --- Basic LLM Invoke ---
# .invoke() sends a message and waits for the full response
# This is the simplest possible LangChain usage

print("=" * 50)
print("BASIC LLM CALL")
print("=" * 50)

response = llm.invoke("Explain what an API is to a 10-year-old in 3 sentences.")

# response is an AIMessage object — content holds the actual text
print("Response type:", type(response).__name__)
print("\nAnswer:")
print(response.content)

BASIC LLM CALL
Response type: AIMessage

Answer:
An API, or Application Programming Interface, is like a messenger between different computer programs that helps them talk to each other. Imagine you're at a restaurant and you want to order food, but you can't just walk into the kitchen and make it yourself - instead, you give your order to the waiter, and they take it to the kitchen staff, who then make your food. An API is like the waiter, taking requests from one program and bringing back the answers or information from another program, so they can work together smoothly.


In [5]:
# --- Streaming Response ---
# .stream() yields tokens as they're generated — great for UI responsiveness
# I was confused about why streaming matters, but imagine a user watching
# text appear live instead of waiting 5 seconds for nothing, then boom — wall of text

print("=" * 50)
print("STREAMING RESPONSE (watch tokens appear live)")
print("=" * 50)
print()

for chunk in llm.stream("Give me 3 fun facts about Python programming language."):
    print(chunk.content, end="", flush=True)  # flush=True forces immediate print

print("\n\n✅ Stream complete!")

STREAMING RESPONSE (watch tokens appear live)

1. **Name Inspiration**: Python is named after the British comedy group Monty Python's Flying Circus. The creator, Guido van Rossum, was a fan of the group and chose the name because it was short, unique, and slightly mysterious.

2. **Easy to Learn**: Python is often considered one of the easiest programming languages to learn, especially for beginners. Its syntax is simple and intuitive, making it a popular choice for introductory programming courses and coding boot camps.

3. **Extensive Libraries**: Python has an vast collection of libraries and frameworks that make it suitable for a wide range of applications, including data science, machine learning, web development, and more. Some popular libraries include NumPy, pandas, and scikit-learn for data analysis, and Django and Flask for web development. This extensive ecosystem is one of the reasons why Python has become a popular choice among developers.

✅ Stream complete!


## 📝 Section 4: Core Component 2 – Prompts / PromptTemplates

Hard-coding prompts is fine for one-off tests, but for real apps you need **reusable templates with variables**.

`ChatPromptTemplate` lets you define a prompt with placeholders (`{variable}`) that get filled in at runtime.  
It also handles the system/human message structure automatically.

I used to manually format strings like `f"You are a {role}. Answer: {question}"` — PromptTemplates do that and much more.

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define a reusable prompt template
# {domain} and {question} are variables we fill in later
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Give concise, accurate answers in 2-3 sentences."),
    ("human", "{question}")
])

print("=" * 50)
print("PROMPT TEMPLATE DEMO")
print("=" * 50)

# See what the formatted messages look like before sending to LLM
formatted = prompt.format_messages(
    domain="machine learning",
    question="What is overfitting?"
)

print("\nFormatted messages:")
for msg in formatted:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")

# Now actually run it through the LLM
chain = prompt | llm | StrOutputParser()

print("\nLLM Response:")
result = chain.invoke({"domain": "machine learning", "question": "What is overfitting?"})
print(result)

# Reuse same template with different variables — this is the real power
print("\n--- Reusing same template for different domain ---")
result2 = chain.invoke({"domain": "Indian history", "question": "Who was Chandragupta Maurya?"})
print(result2)

PROMPT TEMPLATE DEMO

Formatted messages:
  [SystemMessage]: You are an expert in machine learning. Give concise, accurate answers in 2-3 sentences.
  [HumanMessage]: What is overfitting?

LLM Response:
Overfitting occurs when a machine learning model is too complex and learns the noise in the training data, resulting in poor performance on new, unseen data. This happens when the model is overly specialized to the training set and fails to generalize well to other data. As a result, the model's accuracy on the training data is high, but its accuracy on test data is low.

--- Reusing same template for different domain ---
Chandragupta Maurya was the founder of the Mauryan Empire, which was the largest empire in ancient India, ruling from 322 BCE to 298 BCE. With the help of his mentor Chanakya, he overthrew the Nanda dynasty and expanded his empire, conquering much of the Indian subcontinent. He is considered one of the greatest emperors in Indian history, known for his military prowess

## ⛓️ Section 5: Core Component 3 – Chains (LCEL)

LangChain Expression Language (LCEL) uses the `|` pipe operator to connect components.

**Key insight:** Every LangChain component (prompt, LLM, parser, retriever) is a `Runnable`.  
The `|` operator just passes the output of one Runnable as input to the next.

This makes chains incredibly readable and easy to modify. Want to add a step? Just insert another `|` in the pipeline.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("=" * 50)
print("SIMPLE CHAIN (LCEL)")
print("=" * 50)

# Chain 1: Simple Q&A chain
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful tutor. Explain concepts simply."),
    ("human", "{question}")
])

# Build the chain: prompt → LLM → string output
# This is LCEL — the pipe syntax. Clicked for me after the 3rd tutorial 😅
qa_chain = qa_prompt | llm | StrOutputParser()

result = qa_chain.invoke({"question": "What is a neural network? Explain like I'm 15."})
print("\nQ&A Chain Output:")
print(result)

print("\n" + "=" * 50)
print("SEQUENTIAL CHAIN (two-step)")
print("=" * 50)

# Chain 2: Sequential — first generate a topic summary, then create quiz questions
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following topic in exactly 2 sentences."),
    ("human", "Topic: {topic}")
])

quiz_prompt = ChatPromptTemplate.from_messages([
    ("system", "Based on this summary, create 2 multiple choice questions."),
    ("human", "Summary: {summary}")
])

# Step 1: summary chain
summary_chain = summary_prompt | llm | StrOutputParser()

# Step 2: quiz chain — takes summary as input
quiz_chain = quiz_prompt | llm | StrOutputParser()

# Combine: pass topic → get summary → feed to quiz generator
# RunnablePassthrough just passes input through unchanged
full_chain = (
    {"summary": summary_chain, "topic": RunnablePassthrough()}
    | quiz_chain
)

print("\nGenerating quiz on 'Photosynthesis'...")
quiz_result = full_chain.invoke({"topic": "Photosynthesis"})
print(quiz_result)

SIMPLE CHAIN (LCEL)

Q&A Chain Output:
Imagine you're trying to recognize different types of animals, like cats and dogs. A neural network is like a super-smart computer program that helps make decisions, like "is this a cat or a dog?"

It's called a "neural" network because it's inspired by the way our brains work. You see, our brains have billions of tiny cells called neurons that talk to each other to help us think and learn.

A neural network is made up of layers of artificial "neurons" (like tiny computers) that work together to make decisions. Here's how it works:

1. **Input layer**: You show the computer a picture of an animal (like a cat or dog).
2. **Hidden layers**: The computer breaks down the picture into smaller pieces, like edges, shapes, and textures. These layers are like a team of experts that analyze the picture from different angles.
3. **Output layer**: The computer uses all the information from the hidden layers to make a decision, like "this is a cat" or "this is

## 🧠 Section 6: Core Component 4 – Memory

Without memory, every message to your chatbot is a blank slate.  
Ask "What is Python?" → fine. Then ask "What did I just ask about?" → it has no idea.

`RunnableWithMessageHistory` wraps any chain and automatically manages conversation history per session.  
I struggled with this the most — the key thing that confused me was the `session_id` concept.  
Each unique session_id gets its own independent conversation history.

In [8]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# In-memory store: maps session_id → message history
# In production, you'd use Redis or a database here
conversation_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Returns (or creates) the message history for a given session."""
    if session_id not in conversation_store:
        conversation_store[session_id] = InMemoryChatMessageHistory()
        print(f"   [Created new session: {session_id}]")
    return conversation_store[session_id]

# Prompt that includes a placeholder for chat history
# MessagesPlaceholder is the magic — it inserts full message history here
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly and patient tutor named Aria. Remember details the user shares."),
    MessagesPlaceholder(variable_name="chat_history"),  # ← history injected here
    ("human", "{input}")
])

base_chain = memory_prompt | llm | StrOutputParser()

# Wrap chain with memory management
chat_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Session config — use same session_id across turns to maintain context
session_config = {"configurable": {"session_id": "harsha-demo-session"}}

print("=" * 50)
print("MEMORY DEMO – Multi-turn Conversation")
print("=" * 50)

# Turn 1
print("\n[Turn 1]")
print("User: Hi! My name is Harsha and I'm learning about neural networks.")
r1 = chat_with_memory.invoke({"input": "Hi! My name is Harsha and I'm learning about neural networks."}, config=session_config)
print(f"Aria: {r1}")

# Turn 2
print("\n[Turn 2]")
print("User: What's my name?")
r2 = chat_with_memory.invoke({"input": "What's my name again? And what topic was I studying?"}, config=session_config)
print(f"Aria: {r2}")

# Turn 3
print("\n[Turn 3]")
print("User: Can you explain backpropagation briefly?")
r3 = chat_with_memory.invoke({"input": "Can you explain backpropagation briefly?"}, config=session_config)
print(f"Aria: {r3}")

print("\n✅ Memory is working! The assistant remembered the name and topic across turns.")

# Show the stored message history
history = conversation_store["harsha-demo-session"]
print(f"\nTotal messages in history: {len(history.messages)}")

MEMORY DEMO – Multi-turn Conversation

[Turn 1]
User: Hi! My name is Harsha and I'm learning about neural networks.
   [Created new session: harsha-demo-session]
Aria: Hello Harsha, it's nice to meet you. I'm Aria, your friendly tutor. I'd be happy to help you learn about neural networks. What specific aspects of neural networks are you currently exploring or would you like to start with the basics?

[Turn 2]
User: What's my name?
Aria: Your name is Harsha, and you're studying neural networks. I'm keeping track of our conversation, so don't worry if you need to recall anything – I've got it noted down. How can I assist you with neural networks today, Harsha?

[Turn 3]
User: Can you explain backpropagation briefly?
Aria: Backpropagation is a fundamental concept in neural networks, Harsha. It's an algorithm used to train neural networks by minimizing the error between the network's predictions and the actual outputs.

In simple terms, backpropagation works by:

1. Forward pass: The netwo

## 🕵️ Section 7: Core Component 5 & 6 – Agents + Tools

An **Agent** uses the LLM as a reasoning engine — it decides *which tools to call* and in *what order*.  
Unlike a chain (fixed sequence), an agent is dynamic and can loop until it has the answer.

**Tools** are functions the agent can invoke. We'll build 3 custom tools:
1. `calculate` — safe math evaluator
2. `get_current_time` — returns current datetime
3. `text_stats` — counts words/characters in text

The `@tool` decorator is so clean — it automatically extracts the function name, docstring, and parameters to tell the LLM when and how to use it.

In [9]:
from langchain_core.tools import tool
import datetime
import math

# Tool 1: Safe math calculator
# The docstring is CRITICAL — the LLM reads it to decide when to use this tool
@tool
def calculate(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result.
    Input should be a valid math expression like '15 * 0.18' or '(100 + 200) / 3'.
    Supports: +, -, *, /, **, sqrt (use math.sqrt(x)), round(x, n).
    """
    try:
        # Allow only safe math operations — no arbitrary code execution
        safe_globals = {"__builtins__": {}, "math": math, "round": round, "abs": abs}
        result = eval(expression, safe_globals)
        return f"Result: {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"


# Tool 2: Current time
@tool
def get_current_time(query: str) -> str:
    """
    Returns the current date and time in IST (Indian Standard Time).
    Use when the user asks about the current time, date, day, or year.
    Input: any string (it's ignored, just pass 'now').
    """
    now = datetime.datetime.now()
    return f"Current date and time: {now.strftime('%A, %B %d, %Y at %H:%M:%S')}"


# Tool 3: Text statistics
@tool
def text_stats(text: str) -> str:
    """
    Analyzes a given text and returns word count, character count, and sentence count.
    Use when the user wants to know how many words, characters, or sentences are in a text.
    Input: the text to analyze.
    """
    words = len(text.split())
    chars = len(text)
    chars_no_spaces = len(text.replace(" ", ""))
    sentences = len([s for s in text.split('.') if s.strip()])
    return (
        f"Text Statistics:\n"
        f"  Words: {words}\n"
        f"  Characters (with spaces): {chars}\n"
        f"  Characters (no spaces): {chars_no_spaces}\n"
        f"  Sentences (approx): {sentences}"
    )


tools = [calculate, get_current_time, text_stats]

print("✅ Tools defined:")
for t in tools:
    print(f"   - {t.name}: {t.description[:60]}...")

✅ Tools defined:
   - calculate: Evaluates a mathematical expression and returns the result.
...
   - get_current_time: Returns the current date and time in IST (Indian Standard Ti...
   - text_stats: Analyzes a given text and returns word count, character coun...


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Agent prompt must include agent_scratchpad for the agent's reasoning steps
agent_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant with access to tools for calculations, time, and text analysis. "
     "Always use the appropriate tool when the question requires it. "
     "Be concise and clear in your final answers."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # where tool call results get stored
])

# Create the agent — this binds the LLM + tools + prompt together
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# AgentExecutor runs the agent loop: think → call tool → observe → repeat until done
# verbose=True shows us the agent's reasoning steps
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,       # Shows tool calls and reasoning
    max_iterations=5,   # Safety limit — prevents infinite loops
    handle_parsing_errors=True  # Gracefully handles LLM output parse errors
)

print("=" * 50)
print("AGENT DEMO")
print("=" * 50)

# Test 1: Multi-tool query (needs calculate AND time)
print("\n[Query 1: Multi-tool]")
result1 = agent_executor.invoke({
    "input": "What is 18% GST on a bill of ₹4,500? Also, what is today's date?"
})
print("\n✅ Final Answer:", result1["output"])

print("\n" + "-" * 40)

# Test 2: Text analysis
print("\n[Query 2: Text stats]")
result2 = agent_executor.invoke({
    "input": "Analyze this text: 'Machine learning is a subset of artificial intelligence. It enables computers to learn from data without being explicitly programmed.'"
})
print("\n✅ Final Answer:", result2["output"])

## 📄 Section 8: Core Components 7 & 8 – Document Loader + Vector Store + RAG

RAG (Retrieval-Augmented Generation) is one of the most practical LangChain patterns.

**The flow:**
1. Load documents (text, PDF, web pages)
2. Split into chunks (LLMs have context limits)
3. Convert chunks to vector embeddings
4. Store embeddings in Chroma (vector database)
5. At query time: embed the question, find similar chunks, pass them to LLM

This lets the LLM answer questions about *your* documents without hallucinating made-up facts.

I was confused about embeddings at first. Think of it like this: each text chunk becomes a point in a 384-dimensional space, and "similar meaning" = "nearby points".

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Step 1: Create sample documents ──
# In a real app, you'd use PyPDFLoader or TextLoader to load real files
# For this demo, I'm creating documents manually

sample_texts = [
    """LangChain is an open-source framework for building applications powered by large language models (LLMs).
    It provides tools for prompt management, chaining LLM calls, memory, and connecting to external tools.
    LangChain was created by Harrison Chase and first released in October 2022.""",

    """Groq is a company that provides extremely fast inference for LLMs using their custom LPU (Language Processing Unit) hardware.
    Groq offers a free API tier with models like Llama3, Mixtral, and Gemma.
    Groq's LPU can process tokens at speeds exceeding 500 tokens per second, far faster than GPU-based solutions.""",

    """RAG stands for Retrieval-Augmented Generation. It combines a retrieval system with a language model.
    In RAG, documents are converted to embeddings and stored in a vector database.
    At query time, relevant document chunks are retrieved and passed to the LLM as context.
    This allows the LLM to answer questions about specific documents without being retrained.""",

    """Vector databases store high-dimensional vectors (embeddings) and support efficient similarity search.
    ChromaDB is an open-source vector database that runs locally or in the cloud.
    Other popular vector databases include Pinecone, Weaviate, Qdrant, and FAISS.
    Similarity search finds the nearest vectors using algorithms like cosine similarity.""",

    """Innomatics Research Labs is a data science and AI training institute in Hyderabad, India.
    They offer internship programs in Data Science, Machine Learning, and Generative AI.
    The February 2026 cohort focused on LangChain, RAG systems, and LLM application development."""
]

# Convert to Document objects (standard LangChain format)
documents = [
    Document(page_content=text, metadata={"source": f"doc_{i+1}", "chunk": i})
    for i, text in enumerate(sample_texts)
]

print(f"✅ Created {len(documents)} documents")

# ── Step 2: Split documents into smaller chunks ──
# LLMs have context limits, so long documents need to be split
# chunk_size=300 chars, overlap=50 chars (overlap prevents losing context at boundaries)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]  # tries to split at natural boundaries
)

chunks = text_splitter.split_documents(documents)
print(f"✅ Split into {len(chunks)} chunks")

In [ ]:
# ── Step 3: Create embeddings ──
# HuggingFaceEmbeddings uses a free local model (all-MiniLM-L6-v2)
# It downloads the model (~80MB) on first run — totally free, no API key needed!
# I learned: this model produces 384-dimensional embeddings

print("Loading embedding model (first run downloads ~80MB — please wait)...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}  # use "cuda" if you have GPU
)

print("✅ Embedding model loaded")

# Quick test: embed a sentence and check dimensions
test_embedding = embeddings.embed_query("What is LangChain?")
print(f"   Embedding dimensions: {len(test_embedding)} (each text → {len(test_embedding)}D vector)")

In [ ]:
# ── Step 4: Create Chroma vector store ──
# Chroma stores the embeddings locally (no server needed for demo)
# .from_documents() embeds all chunks and stores them

print("Creating Chroma vector store and indexing chunks...")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="langchain_demo_docs",
    persist_directory="./chroma_demo_db"  # saves to disk so we don't re-embed every time
)

print(f"✅ Vector store created with {vectorstore._collection.count()} vectors")

# ── Step 5: Test retrieval ──
# Before building the full RAG chain, let's verify retrieval works
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}  # retrieve top-2 most similar chunks
)

print("\n--- Testing retrieval ---")
query = "What is Groq and how fast is it?"
relevant_docs = retriever.invoke(query)

print(f"Query: '{query}'")
print(f"Retrieved {len(relevant_docs)} relevant chunks:")
for i, doc in enumerate(relevant_docs):
    print(f"\n  [Chunk {i+1} | Source: {doc.metadata['source']}]:")
    print(f"  {doc.page_content[:150]}...")

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# ── Step 6: Build the full RAG chain ──
# RAG prompt: includes retrieved context + user question
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a helpful assistant that answers questions based ONLY on the provided context.
If the context doesn't contain the answer, say 'I don't have information about that in my documents.'
Do not make up answers.

Context:
{context}"""),
    ("human", "{question}")
])

def format_docs(docs):
    """Concatenate retrieved document chunks into a single context string."""
    return "\n\n---\n\n".join([
        f"[Source: {doc.metadata['source']}]\n{doc.page_content}"
        for doc in docs
    ])

# Full RAG chain:
# 1. "context": retrieve relevant docs, format them
# 2. "question": pass through unchanged
# 3. Feed both into rag_prompt → LLM → parse output
rag_chain = (
    {
        "context": retriever | format_docs,  # retrieve + format
        "question": RunnablePassthrough()    # pass question unchanged
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("=" * 50)
print("RAG CHAIN DEMO")
print("=" * 50)

test_questions = [
    "What is LangChain and who created it?",
    "How fast is Groq's LPU?",
    "What is RAG and how does it work?",
    "What is Innomatics Research Labs?",
    "What is the capital of Japan?"  # Not in our docs — should say it doesn't know
]

for i, question in enumerate(test_questions, 1):
    print(f"\n[Q{i}]: {question}")
    answer = rag_chain.invoke(question)
    print(f"[A{i}]: {answer}")
    print("-" * 40)

## 🏗️ Section 9: Architecture Diagram

Here's the full LangChain app architecture — from user input to final output.
Copy this into your blog post or Medium article!

In [31]:
diagram = """
╔══════════════════════════════════════════════════════════════════════╗
║           LANGCHAIN APPLICATION ARCHITECTURE                        ║
║           Harsha Devupalli | Innomatics Internship Feb 2026          ║
╚══════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────┐
  │                     USER INPUT                                  │
  │           "What is 18% GST on ₹4500?"                          │
  └───────────────────────────┬─────────────────────────────────────┘
                              │
                              ▼
  ┌───────────────────────────────────────────────────────────────┐
  │  PROMPT TEMPLATE                                              │
  │  ChatPromptTemplate → formats system + human messages         │
  │  Variables: {input}, {chat_history}, {context}                │
  └───────────────────────────┬───────────────────────────────────┘
                              │
                              ▼
  ┌───────────────────────────────────────────────────────────────┐
  │  CHAT MODEL (LLM)                                             │
  │  ChatGroq → Llama3.3-70B-Versatile                            │
  │  Processes: formatted prompt → generates response             │
  └───────────────────────────┬───────────────────────────────────┘
                              │
              ┌───────────────┴───────────────┐
              │                               │
              ▼                               ▼
  ┌───────────────────────┐     ┌─────────────────────────────┐
  │  DIRECT OUTPUT        │     │  AGENT LOOP                 │
  │  (Simple chains)      │     │  (Tool-calling agents)      │
  │  StrOutputParser()    │     │                             │
  │         │             │     │  ┌──────────────────────┐   │
  │         ▼             │     │  │ Tool: calculate()    │   │
  │   [FINAL ANSWER]      │     │  │ Tool: get_time()     │   │
  └───────────────────────┘     │  │ Tool: text_stats()   │   │
                                │  └──────────┬───────────┘   │
                                │             │               │
                                │             ▼               │
                                │  [Tool Results → LLM]       │
                                │             │               │
                                │             ▼               │
                                │      [FINAL ANSWER]         │
                                └─────────────────────────────┘

  ─────────────────────────────────────────────────────────────────
  RAG PIPELINE (for document Q&A):
  ─────────────────────────────────────────────────────────────────

  [USER QUESTION]
        │
        ▼
  [HuggingFace Embeddings]  →  embed question into vector
        │
        ▼
  [ChromaDB Vector Store]   →  cosine similarity search
        │
        ▼
  [Top-K Relevant Chunks]   →  most similar document sections
        │
        └──────────────┐
                       ▼
  [RAG Prompt: context + question]  →  [ChatGroq LLM]  →  [ANSWER]

  ─────────────────────────────────────────────────────────────────
  MEMORY LAYER (across all flows):
  ─────────────────────────────────────────────────────────────────

  [RunnableWithMessageHistory]
        │
        ├── session_id: "user-123"  →  [InMemoryChatMessageHistory]
        ├── session_id: "user-456"  →  [InMemoryChatMessageHistory]
        └── (in prod: Redis / PostgreSQL)
"""

print(diagram)


╔══════════════════════════════════════════════════════════════════════╗
║           LANGCHAIN APPLICATION ARCHITECTURE                        ║
║           Harsha Devupalli | Innomatics Internship Feb 2026          ║
╚══════════════════════════════════════════════════════════════════════╝

  ┌─────────────────────────────────────────────────────────────────┐
  │                     USER INPUT                                  │
  │           "What is 18% GST on ₹4500?"                          │
  └───────────────────────────┬─────────────────────────────────────┘
                              │
                              ▼
  ┌───────────────────────────────────────────────────────────────┐
  │  PROMPT TEMPLATE                                              │
  │  ChatPromptTemplate → formats system + human messages         │
  │  Variables: {input}, {chat_history}, {context}                │
  └───────────────────────────┬───────────────────────────────────┘
                        

## 📊 Section 10: Component Summary Table

Quick reference for all LangChain components we covered in this notebook.

In [26]:
# Summary table — printed nicely for blog screenshots

summary = [
    ("LLM / ChatGroq",        "ChatGroq",                          "Base language model interface"),
    ("Prompts",               "ChatPromptTemplate",                 "Reusable prompt templates with variables"),
    ("Output Parser",         "StrOutputParser",                   "Converts AIMessage to plain string"),
    ("Chains (LCEL)",         "prompt | llm | parser",             "Composable pipeline with | operator"),
    ("Memory",                "RunnableWithMessageHistory",         "Persists conversation context per session"),
    ("Agents",                "create_tool_calling_agent",          "Dynamic reasoning + tool selection"),
    ("Tools",                 "@tool decorator",                   "Functions the agent can call"),
    ("Document Loader",       "TextLoader / PyPDFLoader",           "Loads files into Document objects"),
    ("Text Splitter",         "RecursiveCharacterTextSplitter",     "Splits docs into small chunks"),
    ("Embeddings",            "HuggingFaceEmbeddings",              "Converts text to numerical vectors"),
    ("Vector Store",          "Chroma",                            "Stores + retrieves embeddings by similarity"),
    ("Retriever",             "vectorstore.as_retriever()",         "Finds relevant chunks for a query"),
]

print("=" * 80)
print(f"{'COMPONENT':<25} {'CLASS/METHOD':<35} {'PURPOSE':<40}")
print("=" * 80)
for component, cls, purpose in summary:
    print(f"{component:<25} {cls:<35} {purpose:<40}")
print("=" * 80)

print("\n✅ All components demonstrated in this notebook!")

COMPONENT                 CLASS/METHOD                        PURPOSE                                 
LLM / ChatGroq            ChatGroq                            Base language model interface           
Prompts                   ChatPromptTemplate                  Reusable prompt templates with variables
Output Parser             StrOutputParser                     Converts AIMessage to plain string      
Chains (LCEL)             prompt | llm | parser               Composable pipeline with | operator     
Memory                    RunnableWithMessageHistory          Persists conversation context per session
Agents                    create_tool_calling_agent           Dynamic reasoning + tool selection      
Tools                     @tool decorator                     Functions the agent can call            
Document Loader           TextLoader / PyPDFLoader            Loads files into Document objects       
Text Splitter             RecursiveCharacterTextSplitter      Splits doc